In [5]:
%pip install h3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 1.8 MB/s eta 0:00:00a 0:00:010m

[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import h3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, monotonically_increasing_id, udf, coalesce
from pyspark.sql.types import StringType

In [3]:
import os
os.environ["HADOOP_USER_NAME"] = "hdfs"
# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("Uber_Medallion_Silver_Layer") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/02 12:15:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# 2. Define H3 Geospatial UDF (Resolution 9)
@udf(returnType=StringType())
def compute_h3(lat, lon, resolution=9):
    if lat is None or lon is None:
        return None
    try:
        # Use h3.latlng_to_cell for h3-py v4+ (fallback to h3.geo_to_h3 if using older versions)
        return h3.latlng_to_cell(lat, lon, resolution)
    except Exception:
        return None

In [5]:
# 3. Load Bronze Data
df_bronze = spark.read.parquet("hdfs://uber-hadoop-master:9000/data/bronze/rides/*.parquet")

df_limited = df_bronze.limit(100000)



In [6]:
# 4. Drop Operational Columns & Add Trip ID
columns_to_drop = [
    "dispatching_base_num", "originating_base_num", "shared_request_flag", 
    "shared_match_flag", "access_a_ride_flag", "wav_request_flag", "wav_match_flag"
]
df_base = df_limited.drop(*columns_to_drop).withColumn("trip_id", monotonically_increasing_id())

In [7]:
# Fill missing on_scene_datetime with the request_datetime
df_base = df_base.withColumn(
    "on_scene_datetime",
    coalesce(col("on_scene_datetime"), col("request_datetime"))
)

In [8]:
# 5. Define Data Quality Filtration Rules
valid_conditions = (
    col("hvfhs_license_num").isNotNull() &
    col("request_datetime").isNotNull() &
    col("pickup_datetime").isNotNull() &
    col("dropoff_datetime").isNotNull() &
    (col("request_datetime") <= col("pickup_datetime")) &
    (col("pickup_datetime") < col("dropoff_datetime")) &
    col("PULocationID").between(1, 265) &
    col("DOLocationID").between(1, 265) &
    (col("trip_time") > 0) &
    (col("trip_miles") >= 0.0) &
    (col("base_passenger_fare") >= 0.0) &
    (col("tolls") >= 0.0) &
    (col("bcf") >= 0.0) &
    (col("sales_tax") >= 0.0) &
    (col("congestion_surcharge") >= 0.0) &
    (col("airport_fee") >= 0.0) &
    (col("tips") >= 0.0) &
    (col("driver_pay") >= 0.0)
)

In [9]:
# 6. Split Data: Clean vs. Quarantine
df_clean = df_base.filter(valid_conditions)
df_bad_records = df_base.filter(~valid_conditions)

In [10]:
# 7. Write Bad Data to Quarantine (Append mode to keep historical bad records)
df_bad_records.write \
    .mode("append") \
    .parquet("hdfs://uber-hadoop-master:9000/data/quarantine/rides/")

[Stage 1:>                                                        (0 + 12) / 15]

26/09/02 12:18:48 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 176412 ms exceeds timeout 120000 ms
26/09/02 12:18:49 WARN SparkContext: Killing executors is not supported by current scheduler.


In [11]:
# 8. Load Real Reference Data
df_zones = spark.read.csv("hdfs://uber-hadoop-master:9000/data/reference/zone_centroids.csv", header=True, inferSchema=True)

In [12]:
# 9. Spatial Joins for Coordinates
df_silver = df_clean.join(
    df_zones.withColumnRenamed("LocationID", "PULocationID") \
            .withColumnRenamed("lat", "start_lat") \
            .withColumnRenamed("lon", "start_lon"),
    on="PULocationID",
    how="left"
).join(
    df_zones.withColumnRenamed("LocationID", "DOLocationID") \
            .withColumnRenamed("lat", "end_lat") \
            .withColumnRenamed("lon", "end_lon"),
    on="DOLocationID",
    how="left"
)

In [13]:
# 10. Apply H3 Hashing
df_silver_final = df_silver \
    .withColumn("start_geo_hash", compute_h3(col("start_lat"), col("start_lon"))) \
    .withColumn("end_geo_hash", compute_h3(col("end_lat"), col("end_lon")))

In [14]:
# 11. Write Cleaned, Enriched Data to Silver Layer
df_silver_final.write \
    .mode("overwrite") \
    .parquet("hdfs://uber-hadoop-master:9000/data/silver/staging_rides_geo/")

In [15]:
from pyspark.sql.functions import col

# 1. Read the data from HDFS
df_silver = spark.read.parquet("hdfs://uber-hadoop-master:9000/data/silver/staging_rides_geo/")

# 2. Limit to 50 rows (rendering millions of rows will freeze your browser)
df_preview = df_silver.limit(50)

# 3. Cast timestamps to strings to bypass the Pandas datetime bug
for field in df_preview.schema.fields:
    if field.dataType.typeName() == 'timestamp':
        df_preview = df_preview.withColumn(field.name, col(field.name).cast("string"))

# 4. Convert to Pandas and display it (leaving the variable at the end renders the table)
df_preview.toPandas()

,DOLocationID,PULocationID,hvfhs_license_num,request_datetime,on_scene_datetime,pickup_datetime,dropoff_datetime,trip_miles,trip_time,base_passenger_fare,...,airport_fee,tips,driver_pay,trip_id,start_lat,start_lon,end_lat,end_lon,start_geo_hash,end_geo_hash
0,158,161,HV0003,2024-01-01 00:21:47,2024-01-01 00:25:06,2024-01-01 00:28:08,2024-01-01 01:05:39,2.830,2251,45.61,...,0.0,0.00,40.18,0,40.765064,-73.985319,40.734519,-74.010269,892a100d65bffff,892a1072137ffff
1,79,137,HV0003,2024-01-01 00:10:56,2024-01-01 00:11:08,2024-01-01 00:12:53,2024-01-01 00:20:05,1.570,432,10.05,...,0.0,0.00,6.12,1,40.739546,-73.977083,40.729269,-73.987361,892a100d2a3ffff,892a1072c97ffff
2,186,79,HV0003,2024-01-01 00:20:04,2024-01-01 00:21:51,2024-01-01 00:23:05,2024-01-01 00:35:16,1.980,731,18.07,...,0.0,0.00,9.47,2,40.729269,-73.987361,40.712800,-74.006000,892a1072c97ffff,892a1072893ffff
3,148,234,HV0003,2024-01-01 00:35:46,2024-01-01 00:39:59,2024-01-01 00:41:04,2024-01-01 00:56:34,1.990,930,17.17,...,0.0,0.00,11.35,3,40.736072,-73.990189,40.715936,-73.986806,892a100d277ffff,892a1072dd3ffff
4,97,148,HV0003,2024-01-01 00:48:19,2024-01-01 00:56:23,2024-01-01 00:57:21,2024-01-01 01:10:02,2.650,761,38.67,...,0.0,0.00,28.63,4,40.715936,-73.986806,40.690771,-73.976624,892a1072dd3ffff,892a100da57ffff
5,95,255,HV0003,2024-01-01 00:03:47,2024-01-01 00:05:53,2024-01-01 00:06:15,2024-01-01 00:27:53,7.020,1298,32.16,...,0.0,0.00,24.35,5,40.712800,-74.006000,40.721415,-73.843827,892a1072893ffff,892a100c5afffff
6,212,95,HV0003,2024-01-01 00:22:51,2024-01-01 00:29:17,2024-01-01 00:29:47,2024-01-01 00:50:08,11.330,1221,45.83,...,0.0,0.00,30.98,6,40.721415,-73.843827,40.712800,-74.006000,892a100c5afffff,892a1072893ffff
7,47,213,HV0003,2024-01-01 00:45:34,2024-01-01 00:57:29,2024-01-01 00:57:50,2024-01-01 01:11:27,3.430,817,23.23,...,0.0,0.00,20.73,7,40.712800,-74.006000,40.712800,-74.006000,892a1072893ffff,892a1072893ffff
8,114,209,HV0003,2024-01-01 00:11:51,2024-01-01 00:15:46,2024-01-01 00:16:00,2024-01-01 00:28:13,1.540,733,15.42,...,0.0,0.00,10.40,8,40.707239,-74.002743,40.730878,-73.998819,892a1072d4bffff,892a1072cd7ffff
9,209,113,HV0003,2024-01-01 00:26:48,2024-01-01 00:33:02,2024-01-01 00:33:15,2024-01-01 00:46:39,1.720,804,13.65,...,0.0,0.00,11.38,9,40.712800,-74.006000,40.707239,-74.002743,892a1072893ffff,892a1072d4bffff


26/09/02 12:34:53 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from uber-jupyter:36705 in 120 seconds
26/09/02 12:39:35 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from uber-jupyter:36705 in 120 seconds
26/09/02 12:39:35 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.rpc.RpcTimeoutException: Futures timed out after [120 seconds]. This timeout is controlled by spark.rpc.askTimeout
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:62)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:58)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:76)
	at org.a